In [17]:
# Run this in terminal, not inside notebook

# !pip uninstall -y transformers sentence-transformers langchain-huggingface
# !pip install torch sentence-transformers==5.0.0 ransformers==4.53.3 chromadb python-dotenv


In [18]:
from dotenv import load_dotenv
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
load_dotenv()

True

In [19]:
# Create LangChain documents for IPL players

doc1 = Document(
        page_content="Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.",
        metadata={"team": "Royal Challengers Bangalore"}
    )
doc2 = Document(
        page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
        metadata={"team": "Mumbai Indians"}
    )
doc3 = Document(
        page_content="MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.",
        metadata={"team": "Chennai Super Kings"}
    )
doc4 = Document(
        page_content="Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.",
        metadata={"team": "Mumbai Indians"}
    )
doc5 = Document(
        page_content="Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.",
        metadata={"team": "Chennai Super Kings"}
    )



In [20]:
docs = [doc1,doc2,doc3,doc4,doc5]

In [21]:
for i, doc in enumerate(docs):
    print(i, repr(doc.page_content))

0 'Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.'
1 "Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure."
2 'MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.'
3 'Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'
4 'Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'


In [22]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001"
)

In [23]:
import shutil

shutil.rmtree("./chroma_db", ignore_errors=True)

In [24]:
vector_store = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

In [25]:
# add documents
vector_store.add_documents(docs)

['31319ba4-7f7d-4dc9-bef1-370809f9812f',
 '8bd784b1-409a-42b8-aecc-6a711207da27',
 '411423e1-175a-403f-a6f5-982cdc7fab5e',
 'b5b3e196-ef59-4cdc-b828-a7d81f1120fe',
 'e36db8a7-4664-450f-bdc3-2e555d16e77c']

In [26]:
# view document
vector_store.get(include=['embeddings','documents','metadatas'])

{'ids': ['2e9411c7-ccd7-4a75-91ae-72b627332492',
  '914a1a06-b4ba-4b5f-a139-c71368ca06f3',
  'a28df385-c4f2-47da-90e1-16e79e3e767a',
  '8b081ed3-d2ff-42e3-8273-723a4cdeafb8',
  '601dd23e-9773-475c-908e-2e1ccf935698',
  'a13c5d5f-0ab6-40c3-b656-1c3cca677b69',
  'c4b28b69-dba4-4207-9061-887b23647d25',
  'b679a257-a979-4a9a-be67-a7896bd1d53e',
  'dddbe311-1d47-45ed-b23d-5e84760dbb9f',
  '5fbaa576-c7d7-446a-8f21-c769ced238d3',
  'add21c78-a244-4cc7-9748-49fb1e6be94a',
  '9bb0a595-8aaa-4ca1-b32c-f00379ab07dc',
  '6d2322ae-3786-4e57-b4fc-9ffdee804533',
  '34b90927-64a6-448f-bf62-64da4534cf9c',
  'f91be5b9-dd5f-401e-8c10-e695cd1089cd',
  '31319ba4-7f7d-4dc9-bef1-370809f9812f',
  '8bd784b1-409a-42b8-aecc-6a711207da27',
  '411423e1-175a-403f-a6f5-982cdc7fab5e',
  'b5b3e196-ef59-4cdc-b828-a7d81f1120fe',
  'e36db8a7-4664-450f-bdc3-2e555d16e77c'],
 'embeddings': array([[-0.00982054,  0.02545763,  0.02402782, ...,  0.01414876,
         -0.01560954, -0.00266117],
        [-0.01989721,  0.01092714,  

In [27]:
# search document
vector_store.similarity_search(
    query='Who among these are a bowler?',
    k=2
)

[Document(metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
 Document(metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.')]

In [28]:
# search with similarity score
vector_store.similarity_search_with_score(
    query='Who among these are a bowler?',
    k=2
)

[(Document(metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
  0.6406819820404053),
 (Document(metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
  0.6406819820404053)]

In [35]:
# meta-data filtering
vector_store.similarity_search_with_score(
    query="Players",
    filter={"team": "Mumbai Indians"},
    k=1
)

[(Document(metadata={'team': 'Mumbai Indians'}, page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure."),
  0.7226663827896118)]

In [36]:
# update documents
updated_doc1 = Document(
    page_content="Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple centuries in a single season. Despite RCB not winning an IPL title under his captaincy, Kohli's passion and fitness set a benchmark for the league. His ability to chase targets and anchor innings has made him one of the most dependable players in T20 cricket.",
    metadata={"team": "Royal Challengers Bangalore"}
)
vector_store.update_document(document_id='2e9411c7-ccd7-4a75-91ae-72b627332492', document=updated_doc1)

In [37]:
# view document
vector_store.get(include=['embeddings','documents','metadatas'])

{'ids': ['2e9411c7-ccd7-4a75-91ae-72b627332492',
  '914a1a06-b4ba-4b5f-a139-c71368ca06f3',
  'a28df385-c4f2-47da-90e1-16e79e3e767a',
  '8b081ed3-d2ff-42e3-8273-723a4cdeafb8',
  '601dd23e-9773-475c-908e-2e1ccf935698',
  'a13c5d5f-0ab6-40c3-b656-1c3cca677b69',
  'c4b28b69-dba4-4207-9061-887b23647d25',
  'b679a257-a979-4a9a-be67-a7896bd1d53e',
  'dddbe311-1d47-45ed-b23d-5e84760dbb9f',
  '5fbaa576-c7d7-446a-8f21-c769ced238d3',
  'add21c78-a244-4cc7-9748-49fb1e6be94a',
  '9bb0a595-8aaa-4ca1-b32c-f00379ab07dc',
  '6d2322ae-3786-4e57-b4fc-9ffdee804533',
  '34b90927-64a6-448f-bf62-64da4534cf9c',
  'f91be5b9-dd5f-401e-8c10-e695cd1089cd',
  '31319ba4-7f7d-4dc9-bef1-370809f9812f',
  '8bd784b1-409a-42b8-aecc-6a711207da27',
  '411423e1-175a-403f-a6f5-982cdc7fab5e',
  'b5b3e196-ef59-4cdc-b828-a7d81f1120fe',
  'e36db8a7-4664-450f-bdc3-2e555d16e77c'],
 'embeddings': array([[-0.00719311,  0.02029932,  0.02837158, ...,  0.01391   ,
         -0.01240376, -0.00345245],
        [-0.01989721,  0.01092714,  

In [38]:
# delete document
vector_store.delete(ids=['2e9411c7-ccd7-4a75-91ae-72b627332492'])

In [39]:
# view document
vector_store.get(include=['embeddings','documents','metadatas'])

{'ids': ['914a1a06-b4ba-4b5f-a139-c71368ca06f3',
  'a28df385-c4f2-47da-90e1-16e79e3e767a',
  '8b081ed3-d2ff-42e3-8273-723a4cdeafb8',
  '601dd23e-9773-475c-908e-2e1ccf935698',
  'a13c5d5f-0ab6-40c3-b656-1c3cca677b69',
  'c4b28b69-dba4-4207-9061-887b23647d25',
  'b679a257-a979-4a9a-be67-a7896bd1d53e',
  'dddbe311-1d47-45ed-b23d-5e84760dbb9f',
  '5fbaa576-c7d7-446a-8f21-c769ced238d3',
  'add21c78-a244-4cc7-9748-49fb1e6be94a',
  '9bb0a595-8aaa-4ca1-b32c-f00379ab07dc',
  '6d2322ae-3786-4e57-b4fc-9ffdee804533',
  '34b90927-64a6-448f-bf62-64da4534cf9c',
  'f91be5b9-dd5f-401e-8c10-e695cd1089cd',
  '31319ba4-7f7d-4dc9-bef1-370809f9812f',
  '8bd784b1-409a-42b8-aecc-6a711207da27',
  '411423e1-175a-403f-a6f5-982cdc7fab5e',
  'b5b3e196-ef59-4cdc-b828-a7d81f1120fe',
  'e36db8a7-4664-450f-bdc3-2e555d16e77c'],
 'embeddings': array([[-0.01989721,  0.01092714,  0.01730603, ...,  0.00973945,
         -0.0176257 , -0.00460104],
        [-0.01358705, -0.00359449,  0.01163961, ...,  0.01149882,
         -0.